# Instructions

In this tutorial, we will perform multi-label classification using an ECG-FM model finetuned on the [MIMIC-IV-ECG v1.0 dataset](https://physionet.org/content/mimic-iv-ecg/1.0/). It outlines the data and model loading, as well as inference, same-sample prediction aggregation, and visualizations for embeddings and saliency maps.

ECG-FM was developed in collaboration with the [fairseq_signals](https://github.com/Jwoo5/fairseq-signals) framework, which implements a collection of deep learning methods for ECG analysis.

This is segment the ECG into inputs of 5 s and perform a label-specific aggregation of the predictions from each sample

This document serves largely as a quickstart introduction. Much of this functionality is also available via the [fairseq-signals scripts](https://github.com/bowang-lab/ECG-FM/blob/main/notebooks/infer_cli.ipynb), as well the [ECG-FM scripts](https://github.com/bowang-lab/ECG-FM/tree/main/scripts).

## Installation

Begin by cloning [fairseq_signals](https://github.com/Jwoo5/fairseq-signals) and refer to the installation section in the top-level README. For example, the following commands are sufficient at the present moment:
```
# Creating `fairseq` environment:
conda create --name fairseq python=3.10.6
source activate fairseq
git clone https://github.com/Jwoo5/fairseq-signals
cd fairseq-signals
python3 -m pip install pip==24.0
python3 -m pip install -e .
```

In [1]:
import os

root = os.path.dirname(os.getcwd())

## Download checkpoints

Checkpoints are available on [HuggingFace](https://huggingface.co/wanglab/ecg-fm). The finetuned model be downloaded using the following command:

In [2]:
import sys
sys.executable
import torch, transformers, numpy as np
print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("numpy", np.__version__)

/home/sagemaker-user/.conda/envs/fairseq/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.10.0+cu128
transformers 4.30.2
numpy 2.2.6


In [3]:
import os
from huggingface_hub import hf_hub_download

_ = hf_hub_download(
    repo_id='wanglab/ecg-fm',
    filename='mimic_iv_ecg_finetuned.yaml',
    local_dir=os.path.join(root, 'ckpts'),
)

# Inference

In [4]:
ckpt_path: str = os.path.join(root, 'ckpts/mimic_iv_ecg_finetuned.pt')
assert os.path.isfile(ckpt_path)

device: str = 'cuda'
batch_size: int = 16
num_workers: int = 0

extract_saliency: bool = True

In [5]:
import os
import pandas as pd


from typing import Any, List

def to_list(obj: Any) -> List[Any]:
    if isinstance(obj, list):
        return obj

    if isinstance(obj, (np.ndarray, set, dict)):
        return list(obj)

    return [obj]


CSV_PATH = "/home/sagemaker-user/files/workspace_files/Foundation_exploration/mimic_iv_ecg_test10k_afib.csv"
WFDB_ROOT = "/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files"

df = pd.read_csv(CSV_PATH)

file_paths = [
    os.path.join(
        WFDB_ROOT,
        str(sp).replace("files/", "", 1)
    )
    for sp in df["source_path"]
]

file_paths[:5]

['/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1051/p10512468/s42341564/42341564',
 '/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1721/p17211916/s40965333/40965333',
 '/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1832/p18323186/s44766763/44766763',
 '/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1002/p10022863/s40441658/40441658',
 '/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1720/p17203862/s41908718/41908718']

## Prepare data

To simplify this tutorial, we have processed a sample of 10 ECGs (14 5s segments) from the [CODE-15% v1.0.0 dataset](https://zenodo.org/records/4916206/) using our [end-to-end data preprocessing pipeline](https://github.com/Jwoo5/fairseq-signals/tree/master/scripts/preprocess/ecg). Its README is also helpful if looking to perform inference using your own dataset, where there are already preprocessing scripts implemented for several public datasets.

In [6]:
from typing import List
from itertools import chain

from scipy.io import loadmat

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader

from ecg_transform.inp import ECGInput, ECGInputSchema
from ecg_transform.sample import ECGMetadata, ECGSample
from ecg_transform.t.base import ECGTransform
from ecg_transform.t.common import (
    HandleConstantLeads,
    LinearResample,
    ReorderLeads,
)
from ecg_transform.t.scale import Standardize
from ecg_transform.t.cut import SegmentNonoverlapping

class ECGFMDataset(Dataset):
    def __init__(
        self,
        schema,
        transforms,
        file_paths,
    ):
        self.schema = schema
        self.transforms = transforms
        self.file_paths = file_paths

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        import wfdb
        import numpy as np

        # WFDB record prefix path, e.g. .../s41420867/41420867
        record_path = self.file_paths[idx]

        record = wfdb.rdrecord(record_path)
        feats = record.p_signal.T  # (12, time)
        org_sample_rate = int(record.fs)

        lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

        # ---- BEGIN SINGLE-LEAD PATCH ----
        # Keep only one lead (Lead II), zero-pad the other 11 leads
        lead_name = 'II'
        lead_ind = lead_names.index(lead_name)

        single_lead = feats[lead_ind:lead_ind+1, :]   # shape: (1, time)
        padded = np.zeros_like(feats)                 # shape: (12, time)
        padded[lead_ind:lead_ind+1, :] = single_lead

        feats = padded
        # ---- END SINGLE-LEAD PATCH ----

        metadata = ECGMetadata(
            sample_rate=org_sample_rate,
            num_samples=feats.shape[1],
            lead_names=lead_names,
            unit=None,
            input_start=0,
            input_end=feats.shape[1],
        )
        metadata.file = record_path

        inp = ECGInput(feats, metadata)
        sample = ECGSample(
            inp,
            self.schema,
            self.transforms,
        )
        source = torch.from_numpy(sample.out).float()

        return source, inp

def collate_fn(inps):
    sample_ids = list(
        chain.from_iterable([[inp[1]] * inp[0].shape[0] for inp in inps])
    )
    return torch.concatenate([inp[0] for inp in inps]), sample_ids

def file_paths_to_loader(
    file_paths: List[str],
    schema: ECGInputSchema,
    transforms: List[ECGTransform],
    batch_size=1,
    num_workers=0,
):
    dataset = ECGFMDataset(
        schema,
        transforms,
        file_paths,
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
        sampler=None,
        shuffle=False,
        collate_fn=collate_fn,
        drop_last=False,
    )

In [7]:
ECG_FM_LEAD_ORDER = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
SAMPLE_RATE = 500
N_SAMPLES = SAMPLE_RATE*5

label_def = pd.read_csv(
    os.path.join(root, 'data/mimic_iv_ecg/labels/label_def.csv'),
     index_col='name',
)
label_names = label_def.index.to_list()
label_names

['Poor data quality',
 'Sinus rhythm',
 'Premature ventricular contraction',
 'Tachycardia',
 'Ventricular tachycardia',
 'Supraventricular tachycardia with aberrancy',
 'Atrial fibrillation',
 'Atrial flutter',
 'Bradycardia',
 'Accessory pathway conduction',
 'Atrioventricular block',
 '1st degree atrioventricular block',
 'Bifascicular block',
 'Right bundle branch block',
 'Left bundle branch block',
 'Infarction',
 'Electronic pacemaker']

In [8]:
AGG_METHODS = {
    'Poor data quality': 'max',
    'Sinus rhythm': 'mean',
    'Premature ventricular contraction': 'max',
    'Tachycardia': 'mean',
    'Ventricular tachycardia': 'max',
    'Supraventricular tachycardia with aberrancy': 'max',
    'Bradycardia': 'mean',
    'Infarction': 'mean',
    'Atrioventricular block': 'mean',
    'Right bundle branch block': 'mean',
    'Left bundle branch block': 'mean',
    'Electronic pacemaker': 'max',
    'Atrial fibrillation': 'mean',
    'Atrial flutter': 'mean',
    'Accessory pathway conduction': 'mean',
    '1st degree atrioventricular block': 'mean',
    'Bifascicular block': 'mean',
}

ECG_FM_SCHEMA = ECGInputSchema(
    sample_rate=SAMPLE_RATE,
    expected_lead_order=ECG_FM_LEAD_ORDER,
    required_num_samples=N_SAMPLES,
)

ECG_FM_TRANSFORMS = [
    ReorderLeads(
        expected_order=ECG_FM_LEAD_ORDER,
        missing_lead_strategy='raise',
    ),
    LinearResample(desired_sample_rate=SAMPLE_RATE),
    HandleConstantLeads(strategy='zero'),
    Standardize(),
    SegmentNonoverlapping(segment_length=N_SAMPLES),
]

loader = file_paths_to_loader(
    file_paths,
    ECG_FM_SCHEMA,
    ECG_FM_TRANSFORMS,
    batch_size=batch_size,
    num_workers=num_workers,
)

## Load model

In [9]:
from typing import Dict, List, Optional, Tuple, Type, Union
from collections import OrderedDict

import numpy as np
import pandas as pd

import torch

from fairseq_signals.models import build_model_from_checkpoint
from fairseq_signals.models.classification.ecg_transformer_classifier import (
    ECGTransformerClassificationModel
)

In [10]:
model: ECGTransformerClassificationModel = build_model_from_checkpoint(
    checkpoint_path=ckpt_path
)

# Forcibly enable the return of attention weights for saliency maps
if extract_saliency:
    model.encoder.encoder.need_weights = extract_saliency
    for layer in model.encoder.encoder.layers:
        layer.need_weights = extract_saliency

model.eval()
model.to(device)

ECGTransformerClassificationModel(
  (encoder): ECGTransformerModel(
    (dropout_input): Dropout(p=0.0, inplace=False)
    (dropout_features): Dropout(p=0.0, inplace=False)
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-11): 12 x TransformerEncoderLayer(
          (self_attn): MultiHeadAttention(
            (dropout): Dropout()
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (dropout1): Dropout(p=0.0, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
          (dropout3): Dropout(p=0.0, inplace=False)
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (f

## Infer

In [11]:
def encoder_out_to_emb(x, device='cpu'):
    # fairseq_signals/models/classification/ecg_transformer_classifier.py
    return torch.div(x.sum(dim=1), (x != 0).sum(dim=1))

def infer(
    model,
    loader,
    device,
    extract_saliency: bool = False,
):
    model.eval()

    inps = []
    sources = []
    logits = []
    embs = []
    saliency = []
    file_names = []

    with torch.no_grad():
        for source, inp in loader:
            source = source.to(device)
            out = model(source=source)
            inps.extend(inp)
            sources.append(source.detach().cpu())
            logits.append(out['out'].detach().cpu())
            embs.append(encoder_out_to_emb(out['encoder_out']).detach().cpu())
            if extract_saliency:
                saliency.append(out['saliency'].detach().cpu())
            file_names.extend([i.meta.file for i in inp])

            del source, out
            torch.cuda.empty_cache()

    # Handle predictions
    pred = torch.sigmoid(torch.concatenate(logits)).numpy()
    pred = pd.DataFrame(pred, columns=label_names, index=file_names)

    results = {
        'inps': inps,
        'sources': torch.concatenate(sources).numpy(),
        'embs': torch.concatenate(embs).numpy(),
        'pred': pred,
    }

    # Handle saliency
    if extract_saliency:
        saliency = torch.concatenate(saliency)
        attn = saliency[:, -1] # Consider only the last attention layer
        results['attn_max'] = attn.max(axis=2).values.squeeze().numpy()

    return results

In [12]:
results = infer(model, loader, device, extract_saliency=False)

In [13]:
pred = results['pred']
print(f"Number of 5 s segment predictions: {len(pred)}.")
pred

Number of 5 s segment predictions: 20000.


,Poor data quality,Sinus rhythm,Premature ventricular contraction,Tachycardia,Ventricular tachycardia,Supraventricular tachycardia with aberrancy,Atrial fibrillation,Atrial flutter,Bradycardia,Accessory pathway conduction,Atrioventricular block,1st degree atrioventricular block,Bifascicular block,Right bundle branch block,Left bundle branch block,Infarction,Electronic pacemaker
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1051/p10512468/s42341564/42341564,0.998960,0.954494,0.620812,0.078820,0.020272,0.007584,0.075404,0.023069,0.083859,0.093291,0.008502,0.007832,8.114793e-04,0.026920,0.005644,0.682837,0.010342
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1051/p10512468/s42341564/42341564,0.999323,0.680375,0.331406,0.193599,0.886094,0.040924,0.171991,0.725256,0.037094,0.561021,0.017256,0.006655,1.098478e-03,0.023219,0.018566,0.765959,0.002989
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1721/p17211916/s40965333/40965333,0.927308,0.004914,0.612841,0.980606,0.985369,0.370032,0.992586,0.177061,0.029873,0.998910,0.061455,0.031263,8.491075e-03,0.031423,0.012584,0.744124,0.005412
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1721/p17211916/s40965333/40965333,0.960977,0.004022,0.456027,0.964650,0.999436,0.300827,0.982923,0.620540,0.061849,0.991869,0.136250,0.100365,3.260188e-02,0.034938,0.005407,0.772549,0.079007
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1832/p18323186/s44766763/44766763,0.997480,0.000300,0.183361,0.977291,0.998984,0.091842,0.998260,0.156184,0.006922,0.999690,0.088703,0.001209,9.147089e-08,0.000503,0.003229,0.648405,0.000287
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1938/p19385108/s42439567/42439567,0.990896,0.011130,0.610641,0.917873,0.961456,0.115442,0.665789,0.462411,0.020853,0.880036,0.883286,0.876034,1.300142e-03,0.037460,0.018627,0.813335,0.032957
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1884/p18849711/s45571703/45571703,0.914495,0.002413,0.868787,0.988454,0.999939,0.433793,0.988723,0.740600,0.047486,0.994903,0.131093,0.132748,9.870877e-04,0.005865,0.308811,0.980885,0.227338
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1884/p18849711/s45571703/45571703,0.904513,0.002291,0.835295,0.983768,1.000000,0.510859,0.985734,0.732376,0.095841,0.991640,0.243242,0.264596,8.246750e-03,0.019053,0.650967,0.935914,0.280329
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1779/p17797856/s49541450/49541450,0.999367,0.690955,0.535492,0.953992,0.218565,0.602827,0.009731,0.740443,0.001647,0.179364,0.018654,0.000135,1.411989e-04,0.011445,0.003061,0.507836,0.000278


# Result handling

## Prediction aggregation

In [14]:
pred_agg = pred.groupby(pred.index).agg(AGG_METHODS).astype(float)
print(f"Number of sample-aggregated predictions: {len(pred_agg)}.")
pred_agg

Number of sample-aggregated predictions: 10000.


,Poor data quality,Sinus rhythm,Premature ventricular contraction,Tachycardia,Ventricular tachycardia,Supraventricular tachycardia with aberrancy,Bradycardia,Infarction,Atrioventricular block,Right bundle branch block,Left bundle branch block,Electronic pacemaker,Atrial fibrillation,Atrial flutter,Accessory pathway conduction,1st degree atrioventricular block,Bifascicular block
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1000/p10000635/s43522917/43522917,0.988255,0.936727,0.356922,0.046283,2.091853e-01,0.020989,0.958262,0.493848,0.017388,0.182303,0.014063,0.030263,0.039252,0.104737,0.063201,0.032899,0.017344
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1000/p10000635/s48339811/48339811,0.996571,0.367840,0.453683,0.517893,1.911291e-02,0.207346,0.490883,0.653440,0.060934,0.065584,0.004078,0.169965,0.729861,0.646431,0.752886,0.070030,0.039517
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1000/p10002559/s41675681/41675681,0.999827,0.903440,0.296827,0.096950,2.620646e-03,0.002971,0.008308,0.696426,0.018759,0.042149,0.000751,0.002483,0.057344,0.094884,0.238991,0.014776,0.001053
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1000/p10003019/s44989970/44989970,0.995243,0.085315,0.491857,0.615251,8.806525e-01,0.599856,0.005109,0.669078,0.407841,0.075086,0.005400,0.009751,0.652632,0.994236,0.939913,0.125923,0.195719
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1000/p10003019/s46272123/46272123,0.993017,0.988018,0.366263,0.002157,3.924550e-09,0.002512,0.213146,0.529716,0.000705,0.091738,0.025736,0.038436,0.008379,0.000007,0.009364,0.012256,0.253947
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1998/p19989126/s41616922/41616922,0.997042,0.995320,0.234999,0.008560,2.026688e-04,0.000732,0.974946,0.279668,0.000531,0.003324,0.005766,0.026099,0.015145,0.004795,0.005867,0.006869,0.001157
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1998/p19989183/s49667577/49667577,0.998271,0.537088,0.196228,0.060244,7.630522e-05,0.004670,0.011409,0.260144,0.015871,0.001543,0.000629,0.002436,0.224233,0.570256,0.354220,0.005101,0.000780
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1999/p19993776/s45691497/45691497,0.983067,0.282344,0.626480,0.416940,9.993147e-01,0.420318,0.075831,0.464991,0.093039,0.139014,0.206065,0.726684,0.805801,0.452591,0.851091,0.116493,0.010379
/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files/p1999/p19996783/s45159013/45159013,0.999277,0.467634,0.344593,0.956876,1.727740e-03,0.214207,0.012496,0.794025,0.404233,0.001446,0.005124,0.014199,0.153270,0.965417,0.904609,0.030090,0.000041


In [15]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score

CSV_PATH = "/home/sagemaker-user/files/workspace_files/Foundation_exploration/mimic_iv_ecg_test10k_afib.csv"
WFDB_ROOT = "/home/sagemaker-user/files/workspace_files/MIMIC/MIMIC-ECG/physionet.org/files/mimic-iv-ecg/1.0/files"

df_gt = pd.read_csv(CSV_PATH)

df_gt["file"] = df_gt["source_path"].apply(
    lambda sp: os.path.join(WFDB_ROOT, str(sp).replace("files/", "", 1))
)

df_gt["afib"] = df_gt.iloc[:, 4].astype(int)

df_gt = df_gt.set_index("file")

pred_col = "Atrial fibrillation"
common = pred_agg.index.intersection(df_gt.index)

print("Evaluation samples:", len(common))

if len(common) == 0:
    print("No overlapping samples between pred_agg.index and CSV ground truth index.")
    print("\npred_agg sample index:")
    print(pred_agg.index[:5].tolist())
    print("\nground truth sample index:")
    print(df_gt.index[:5].tolist())
else:
    y_true = df_gt.loc[common, "afib"].values
    y_prob = pred_agg.loc[common, pred_col].values
    y_pred = (y_prob >= 0.5).astype(int)

    accuracy = (y_pred == y_true).mean()
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print("Evaluated label:", pred_col)
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1:       {f1:.4f}")

    if len(np.unique(y_true)) >= 2:
        auroc = roc_auc_score(y_true, y_prob)
        auprc = average_precision_score(y_true, y_prob)
        print(f"AUROC:    {auroc:.4f}")
        print(f"AUPRC:    {auprc:.4f}")
    else:
        print("AUROC/AUPRC skipped because y_true has only one class.")

Evaluation samples: 10000
Evaluated label: Atrial fibrillation
Accuracy: 0.8977
F1:       0.6602
AUROC:    0.9843
AUPRC:    0.9009


## Visualizing embeddings

In [16]:
source, inp = next(iter(loader))

print("Source shape:", source.shape)
print("Lead energy:", torch.sum(torch.abs(source), dim=2)[0])

Source shape: torch.Size([32, 12, 2500])
Lead energy: tensor([   0.0000, 1913.4805,    0.0000,    0.0000,    0.0000,    0.0000,
           0.0000,    0.0000,    0.0000,    0.0000,    0.0000,    0.0000])


In [17]:
# Count predicted 0s and 1s
num_pred_1 = int((y_pred == 1).sum())
num_pred_0 = int((y_pred == 0).sum())

print("Total samples:", len(y_pred))
print("Predicted 1s:", num_pred_1)
print("Predicted 0s:", num_pred_0)
print("Predicted positive rate:", num_pred_1 / len(y_pred))
print("Predicted negative rate:", num_pred_0 / len(y_pred))

Total samples: 10000
Predicted 1s: 1990
Predicted 0s: 8010
Predicted positive rate: 0.199
Predicted negative rate: 0.801


In [18]:
tp = int(((y_true == 1) & (y_pred == 1)).sum())
tn = int(((y_true == 0) & (y_pred == 0)).sum())
fp = int(((y_true == 0) & (y_pred == 1)).sum())
fn = int(((y_true == 1) & (y_pred == 0)).sum())

print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)

TP: 994
TN: 7983
FP: 996
FN: 27
